# Convex Quadratic Knapsack Problem using Frank Wolfe (CQKP_FW)

In [15]:
import numpy as np
from numpy.linalg import eigh
from scipy.sparse import diags
from scipy.linalg import eigvalsh
import cvxpy as cp

In [144]:
def gen_CQKP(n, actv=0.5, rank=1.1, ecc=0.99, seed=0, lmin = 0, lmax = 4, umin=8, umax=12, cut_through_box=True):
    # Generate a random instance of the Convex Quadratic Knapsack Problem (CQKP)
    if not isinstance(n, int) or n <= 0:
        raise ValueError("n must be a positive integer")
    if not (0 <= actv <= 1):
        raise ValueError("actv must be in [0, 1]")
    if rank <= 0:
        raise ValueError("rank must be > 0")
    if not (0 <= ecc < 1):
        raise ValueError("ecc must be in [0, 1)")
    if lmin < 0:
        raise ValueError("lmin must be >= 0")
    if lmax <= lmin:
        raise ValueError("lmax must be > lmin")
    if lmax > umin:
        raise ValueError("lmax must be < umin")
    if umax <= umin:
        raise ValueError("umax must be > umin")
    
    np.random.seed(seed)
    
    # Generate u
    l = lmin + (lmax - lmin) * np.random.rand(n)
    u = umin + (umax - umin) * np.random.rand(n)

    # Generate Q
    G = np.random.rand(round(rank * n), n)
    Q = G.T @ G
    
    # Assume Q is given and ecc (eccentricity) is given
    # Compute eigenvalue decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(Q)  # Sorted by increasing eigenvalue

    # Check if the smallest eigenvalue is greater than "0"
    if eigenvalues[0] > 1e-14:
        # Modify eccentricity only if lambda_n > 0, for when lambda_n = 0 the
        # eccentricity is 1 by default

        lambda_n = eigenvalues[0] # Smallest eigenvalue
        lambda_1 = eigenvalues[-1] # Largest eigenvalue

        # Calculate new eigenvalues with the desired eccentricity
        new_eigenvalues = lambda_n + ((eigenvalues - lambda_n) / (lambda_1 - lambda_n)) * (2 * ecc / (1 - ecc)) * lambda_n
        # lambd = lambda_n * np.ones(n) + (lambda_n / (lambda_1 - lambda_n)) * \
        #         (2 * ecc / (1 - ecc)) * \
        #         ((eigenvalues - lambda_n)  * np.ones(n))
        # Construct the diagonal matrix of the new eigenvalues
        D_new = np.diag(new_eigenvalues)

        # Recompute Q with modified eccentricity
        Q = eigenvectors @ D_new @ eigenvectors.T
    
    # Generate q
    z = np.zeros(n)
    is_out_of_bound = np.random.rand(n) <= actv
    is_out_lr = np.random.rand(n) <= 0.5
    is_out_left = is_out_of_bound & is_out_lr
    is_out_right = is_out_of_bound & ~is_out_lr
    z[is_out_left] = - np.random.rand(sum(is_out_left)) * l[is_out_left]
    z[is_out_right] = u[is_out_right] * (1 + np.random.rand(sum(is_out_right)))
    z[is_out_of_bound] = np.random.rand(sum(is_out_of_bound)) * u[is_out_of_bound]
    
    q = -Q @ z

    # Generate a and b
    a = np.random.rand(n)
    if cut_through_box:
        # Choose 'b' s.t. the hyperplane a^T x = b cuts through the box [l, u].
        # point_in_the_box = np.random.uniform(l, u, n)
        r = np.random.rand()
        b = r * (np.dot(a.T, l)) + (1-r)*(np.dot(a.T, u))
        # b = np.dot(a, (l+u)/2)  # This ensures the hyperplane passes through the center of the box
    else:
        b = np.random.uniform(10e-3, 10e+3, n)
    
    
    return {"Q": Q, "q": q, "l": l, "u": u, "a": a, "b": b}

In [145]:
p = gen_CQKP(10, actv=0.5, rank=1.1, ecc=0.99, seed=0, lmin = 0, lmax = 4, umin=8, umax=12, cut_through_box=True)
p

{'Q': array([[0.94475897, 0.75001258, 0.95861539, 0.79029967, 0.56033056,
         0.56632064, 0.55422351, 0.55296073, 0.72487981, 0.58552698],
        [0.75001258, 0.86404293, 0.80647471, 0.7247639 , 0.53922693,
         0.58302944, 0.64840966, 0.55369333, 0.87236988, 0.6137945 ],
        [0.95861539, 0.80647471, 1.27596885, 0.74407745, 0.67331683,
         0.67063433, 0.74878203, 0.56468229, 0.89600576, 0.64498978],
        [0.79029967, 0.7247639 , 0.74407745, 0.92186892, 0.47619655,
         0.56391525, 0.58549794, 0.5268874 , 0.76465337, 0.6689782 ],
        [0.56033056, 0.53922693, 0.67331683, 0.47619655, 0.59768277,
         0.38137952, 0.48132846, 0.28015438, 0.57109251, 0.45725073],
        [0.56632064, 0.58302944, 0.67063433, 0.56391525, 0.38137952,
         0.59023301, 0.4727686 , 0.46001035, 0.64694314, 0.53928533],
        [0.55422351, 0.64840966, 0.74878203, 0.58549794, 0.48132846,
         0.4727686 , 0.82917347, 0.41507164, 0.83737704, 0.59602141],
        [0.55296073, 0

In [94]:
def is_positive_semidefinite(Q):
    """
    Check if a matrix Q is positive semidefinite (PSD).
    
    Args:
    - Q (numpy.ndarray): A square numpy matrix.
    
    Returns:
    - bool: True if Q is PSD, False otherwise.
    """
    # Check if the matrix is symmetric, as all PSD matrices must be.
    if not np.allclose(Q, Q.T, atol=1e-8):
        return False
    
    # Compute eigenvalues
    eigenvalues = np.linalg.eigvalsh(Q)
    
    # Check if all eigenvalues are non-negative
    return np.all(eigenvalues >= -1e-8)

print("Is Q positive semidefinite?", is_positive_semidefinite(p["Q"]))

Is Q positive semidefinite? True


In [110]:
def linear_subproblem(c, a, b, l, u, verbose=False):
    n = len(c)
    x_star = []
    for i in range(n):
        if c[i] >= 0:
            x_star.append(l[i])
        else:
            x_star.append(u[i])
    
    if (a @ x_star) >= b:
        if verbose:
            print("Feasible solution in a corner of the hyperbox")
        return x_star
    
    # The minimum value of the objective function is not in feasible region
    x_pack = [(i, x_star[i], c[i], a[i]) for i in range(n)]
    # Order in ascending order of c_i/a_i
    x_pack = sorted(x_pack, key=lambda x: x[2] / x[3], reverse=False)
    ordered_index = [t[0] for t in x_pack]
    
    for idx in ordered_index:
        x_star[idx] = u[idx]
        
        if (a @ x_star) > b:
            if verbose:
                print("Feasible solution NOT in a corner of the hyperbox")
            x_star[idx] = (b - (a @ x_star) + (a[idx] * x_star[idx])) / a[idx]
            return x_star
        
        
    if verbose:
        print("No feasible solution found")
    return x_star

In [111]:
def CQKP_FW(CQKP, t=0, eps=1e-6, MaxIter=1000, verbose=True):
    """
    Apply the (possibly, stabilized) Franke-Wolfe algorithm with exact line
    search to the convex Box-Constrained Quadratic program
    
    (P) min { (1/2) x^T * Q * x + q * x : ax >= b, l <= x <= u }
    """
    # Initializations
    n = CQKP['q'].shape[0]
    x = CQKP['u'] / 2  # Start from the middle of the box
    best_lower_bound = -np.inf  # Best lower bound so far

    if verbose:
        print("Frank-Wolfe method")
        print("iter\tf(x)\t\tlower_bound\t\tgap\n")

    i = 1

    while True:
        # Compute function value and direction = - gradient
        v = 0.5 * np.dot(x.T, np.dot(CQKP['Q'], x)) + np.dot(CQKP['q'].T, x)
        g = np.dot(CQKP['Q'], x) + CQKP['q']

        # Solve min { <g, z> : az >= b, l <= z <= u }
        x_bar = linear_subproblem(g, CQKP['a'], CQKP['b'], CQKP['l'], CQKP['u'], verbose=False)

        # Compute the lower bound
        lower_bound = v + np.dot(g.T, x_bar - x)
        if lower_bound > best_lower_bound:
            best_lower_bound = lower_bound

        # Compute the relative gap
        gap = (v - best_lower_bound) / max(abs(v), 1)

        if verbose:
            print(f"{i}\t{v:.8e}\t{best_lower_bound:.8e}\t{gap:.4e}")

        # Stopping criteria
        if gap <= eps:
            status = 'optimal'
            break
        if i > MaxIter:
            status = 'stopped'
            break

        # In the stabilized case, restrict x_bar in the box
        if t > 0:
            x_bar = np.maximum(x - t * CQKP['u'], np.minimum(x + t * CQKP['u'], x_bar))

        # Compute step size
        d = x_bar - x
        den = np.dot(d.T, np.dot(CQKP['Q'], d))
        if den <= 1e-16:
            alpha = 1  # F is linear along d, take max possible stepsize
        else:
            alpha = min(-np.dot(g.T, d) / den, 1)

        # Compute new point
        x = x + alpha * d
        i += 1

    if verbose:
        print(f"stop: {i} iter, status = {status}, fbest = {v:.8e}, gap = {gap:.4e}")

    return v, x, status


In [146]:
problem = gen_CQKP(10, actv=0.5, rank=1.1, ecc=0.99, seed=13, lmin = 0, lmax = 4, umin=8, umax=12, cut_through_box=True)
CQKP_FW(problem, eps=1e-6, MaxIter=1000, verbose=True)

Frank-Wolfe method
iter	f(x)		lower_bound		gap

1	2.30253368e+01	-7.24137395e+00	1.3145e+00
2	4.46903818e-01	1.18222809e-01	3.2868e-01
3	3.89817588e-01	3.89817588e-01	8.8818e-16
stop: 3 iter, status = optimal, fbest = 3.89817588e-01, gap = 8.8818e-16


(0.3898175875168448,
 array([3.11080964, 0.95016488, 3.29711413, 3.86299679, 3.89040446,
        4.12203772, 3.51766525, 3.10210606, 2.56645338, 2.88807292]),
 'optimal')

In [101]:
# (P) min { (1/2) x^T * Q * x + q * x : ax >= b, l <= x <= u }

# Define variables
Q = problem['Q']    
q = problem['q']
a = problem['a']
b = problem['b']
l = problem['l']
u = problem['u']
n = len(q)

x = cp.Variable(n)

# Define constraints
constraints = [a @ x >= b, x >= l, x <= u]

# Define objective
objective = cp.Minimize(0.5 * cp.quad_form(x, Q) + q.T @ x)

# Define problem
P = cp.Problem(objective, constraints)
P.solve(eps_abs=1e-6)

# Print result
print(f"Optimal value: {P.value}")
print(f"Optimal x: {x.value}")
print(f"Status: {P.status}")

Optimal value: -2.3343417382196066
Optimal x: [3.11080964 0.95016488 3.29711413 3.86299679 3.89040446 1.81379699
 2.43616985 3.10210606 2.56645338 2.88807292]
Status: optimal


In [147]:
tries = 1000
failed = 0

for i in range(0, tries):
    np.random.seed(i)
    # random_bool = np.random.rand() < 0.5
    random_bool = True
    data = gen_CQKP(100, actv=0.4, rank=1.1, ecc=0.99, seed=i, lmin = 1, lmax = 5, umin=8, umax=12, cut_through_box=random_bool)

    # Define variables
    Q = data['Q']
    q = data['q']
    a = data['a']
    b = data['b']
    l = data['l']
    u = data['u']
    n = len(q)

    x = cp.Variable(n)
    # Define constraints
    constraints = [a @ x >= b, x >= l, x <= u]

    # Define objective
    objective = cp.Minimize(0.5 * cp.quad_form(x, Q) + q.T @ x)

    # Define problem
    P = cp.Problem(objective, constraints)
    P.solve(eps_abs=1e-6)
    gold_solution = [P.value, x.value]
    y, x_opt, status = CQKP_FW(data, verbose=False, eps=1e-6, MaxIter=1000)
    our_solution = [y, x_opt]

    if not np.allclose(gold_solution[0], our_solution[0], atol=1e-6): # or not np.allclose(gold_solution[1], our_solution[1], atol=1e-6):
        print(f"Test {i} failed")
        print(f"Gold solution: {gold_solution}, status: {P.status}")
        print(f"Our solution: {our_solution}, status: {status}")
        failed += 1

if failed == 0:
    print(f"All {tries} tests passed")
else:
    print(f"{failed} out of {tries} tests failed")
    

Test 1 failed
Gold solution: [5171.969222951483, array([ 2.66808802,  3.88129797, 11.5437684 ,  2.20933029, 11.6341406 ,
        1.36935438,  1.74504085, 10.08583622,  2.5870699 ,  3.15526694,
        2.67677806,  3.740878  ,  1.817809  , 10.78727265,  1.10955037,
       11.02185221,  2.66921921, 11.69209814,  1.56154775,  8.49708385,
        8.07952054,  4.8730463 ,  2.25369671,  3.76929046,  4.50555661,
        4.57842665, 10.21128791,  1.15621913,  1.67932168,  4.51257001,
        1.39338734,  2.6844305 ,  4.83155812,  3.13266114,  3.76750846,
        2.26206252,  3.74600371,  4.33850269,  1.07315311, 10.98848657,
       10.22496094,  3.99266262,  8.23967076,  8.48537382,  1.41290403,
        8.42997652,  8.90283735,  2.17445659,  2.15110135,  1.52011429,
        1.07746783,  3.71534213, 10.27240185,  2.06218664,  2.96629264,
        1.21345018,  3.29647042, 10.32543571, 11.88007996,  3.79903344,
        1.40933772,  9.97507886,  3.77760063,  2.65671708,  1.19981384,
        3.14358

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/cvxpy/problems/problem.py:1403: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


Test 19 failed
Gold solution: [19820.24988756511, array([11.27677184, 11.26049592,  9.3298877 , 10.69635657, 11.92448628,
       11.83443505, 10.67092307, 11.57132835,  9.3795757 , 10.51848441,
        8.74318635,  9.3827408 , 11.93500152, 10.99742764, 11.32443204,
        8.56600071,  8.05691875, 11.42837681, 10.84454023, 11.42545623,
       11.4365206 , 11.1747073 ,  9.89970635,  8.36680076,  9.2500761 ,
       10.55514877, 10.33465194,  9.06389348,  8.21965518, 11.63978983,
        9.21252523,  9.3177263 ,  8.21741839,  8.11017294,  8.49739041,
        8.39520872,  8.64378399,  8.59767051,  9.24120104, 10.23135814,
        9.3857083 ,  9.39633476, 10.45274821,  8.2733001 , 10.38205943,
        9.24040984,  9.00353296, 10.10044375, 10.22267814, 10.37739441,
       10.41790264,  8.29362315, 10.22878002,  9.54291668,  8.82518742,
        9.5169807 ,  8.7284774 ,  9.93953392,  8.24118134,  8.16956824,
       10.66435927,  8.27924541, 10.37042225, 10.97876486, 11.38613341,
       10.0394

KeyboardInterrupt: 